# Artificial Neural Network

### Importing the libraries

In [1]:
import pandas as pd
import numpy as np

import optuna

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD, Adam, RMSprop, Adagrad, Adadelta, Nadam

from sklearn.metrics import roc_auc_score

In [2]:
tf.__version__

'2.15.0'

## Part 1 - Data Preprocessing

### Importing the dataset

In [3]:
data = pd.read_csv('churn_modelling.csv')

data

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [4]:
data.describe(include='all')

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.00000,1.000000e+04,10000,10000.000000,10000,10000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
unique,NaN,NaN,2932,NaN,3,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,Smith,NaN,France,Male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,32,NaN,5014,5457,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,5000.50000,1.569094e+07,NaN,650.528800,NaN,NaN,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,NaN,96.653299,NaN,NaN,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,NaN,350.000000,NaN,NaN,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,NaN,584.000000,NaN,NaN,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,NaN,652.000000,NaN,NaN,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,NaN,718.000000,NaN,NaN,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000


In [5]:
data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)

In [6]:
data = pd.get_dummies(data, drop_first=True)

data

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,0,0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,1,0
2,502,42,8,159660.80,3,1,0,113931.57,1,0,0,0
3,699,39,1,0.00,2,0,0,93826.63,0,0,0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,39,5,0.00,2,1,0,96270.64,0,0,0,1
9996,516,35,10,57369.61,1,1,1,101699.77,0,0,0,1
9997,709,36,7,0.00,1,0,1,42085.58,1,0,0,0
9998,772,42,3,75075.31,2,1,0,92888.52,1,1,0,1


In [7]:
targets = data['Exited']

inputs = data.drop(['Exited'],axis=1)

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(inputs)

scaled = scaler.transform(inputs)

inputs_scaled = pd.DataFrame(scaled, columns=inputs.columns)

inputs_scaled

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
0,-0.326221,0.293517,-1.041760,-1.225848,-0.911583,0.646092,0.970243,0.021886,-0.578736,-0.573809,-1.095988
1,-0.440036,0.198164,-1.387538,0.117350,-0.911583,-1.547768,0.970243,0.216534,-0.578736,1.742740,-1.095988
2,-1.536794,0.293517,1.032908,1.333053,2.527057,0.646092,-1.030670,0.240687,-0.578736,-0.573809,-1.095988
3,0.501521,0.007457,-1.387538,-1.225848,0.807737,-1.547768,-1.030670,-0.108918,-0.578736,-0.573809,-1.095988
4,2.063884,0.388871,-1.041760,0.785728,-0.911583,0.646092,0.970243,-0.365276,-0.578736,1.742740,-1.095988
...,...,...,...,...,...,...,...,...,...,...,...
9995,1.246488,0.007457,-0.004426,-1.225848,0.807737,0.646092,-1.030670,-0.066419,-0.578736,-0.573809,0.912419
9996,-1.391939,-0.373958,1.724464,-0.306379,-0.911583,0.646092,0.970243,0.027988,-0.578736,-0.573809,0.912419
9997,0.604988,-0.278604,0.687130,-1.225848,-0.911583,-1.547768,0.970243,-1.008643,-0.578736,-0.573809,-1.095988
9998,1.256835,0.293517,-0.695982,-0.022608,0.807737,0.646092,-1.030670,-0.125231,1.727904,-0.573809,0.912419


### Splitting the dataset into the Training set and Test set

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(inputs_scaled, targets, test_size=0.2, random_state=42)

## Part 2 - Building the ANN

### Initializing the ANN

In [10]:
ann = Sequential()

### Adding the input layer and the first hidden layer

In [11]:
ann.add(Dense(units=6, activation='relu'))

### Adding the second hidden layer

In [12]:
ann.add(Dense(units=6, activation='relu'))

### Adding the output layer

In [13]:
ann.add(Dense(units=1, activation='sigmoid'))

## Part 3 - Training the ANN

### Compiling the ANN

In [14]:
ann.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['AUC'])

### Training the ANN on the Training set

In [15]:
ann.fit(X_train, y_train, batch_size = 32, epochs = 100)

Epoch 1/100
250/250 [==============================] - 0s 763us/step - loss: 0.5409 - auc: 0.6430
Epoch 2/100
250/250 [==============================] - 0s 647us/step - loss: 0.4588 - auc: 0.7260
Epoch 3/100
250/250 [==============================] - 0s 724us/step - loss: 0.4399 - auc: 0.7547
Epoch 4/100
250/250 [==============================] - 0s 723us/step - loss: 0.4315 - auc: 0.7661
Epoch 5/100
250/250 [==============================] - 0s 608us/step - loss: 0.4259 - auc: 0.7726
Epoch 6/100
250/250 [==============================] - 0s 775us/step - loss: 0.4215 - auc: 0.7769
Epoch 7/100
250/250 [==============================] - 0s 615us/step - loss: 0.4180 - auc: 0.7797
Epoch 8/100
250/250 [==============================] - 0s 741us/step - loss: 0.4154 - auc: 0.7812
Epoch 9/100
250/250 [==============================] - 0s 685us/step - loss: 0.4131 - auc: 0.7835
Epoch 10/100
250/250 [==============================] - 0s 618us/step - loss: 0.4114 - auc: 0.7847
Epoch 11/100
250/25

In [26]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score

def evaluate(model, X_train, y_train, X_test, y_test):
    
    '''Predictions and probabilities for the training set'''
    
    y_train_prob = model.predict(X_train)

    '''Predictions and probabilities for the test set'''
    
    y_test_prob = model.predict(X_test)

    '''Calculate metrics for the training set''' 
    
    roc_train_prob = roc_auc_score(y_train, y_train_prob)
    gini_train_prob = roc_train_prob * 2 - 1
    

    '''Calculate metrics for the test set'''
    
    roc_test_prob = roc_auc_score(y_test, y_test_prob)
    gini_test_prob = roc_test_prob * 2 - 1
    

    results = pd.DataFrame({
        'Dataset': ['Train', 'Test'],
        'Gini': [gini_train_prob * 100, gini_test_prob * 100],
    
    })

    return results

In [27]:
evaluate(ann, X_train, y_train, X_test, y_test)

63/63 [==============================] - 0s 262us/step


,Dataset,Gini
0,Train,73.713785
1,Test,72.224571


## Part 4 - Making the predictions and evaluating the model

### Predicting the result of a single observation

In [36]:
data_test = data.sample(100)

data_test

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
2685,2686,15672115,Lettiere,679,France,Male,60,6,0.00,2,1,1,77331.77,0
8017,8018,15631406,Munro,459,Germany,Male,50,5,109387.90,1,1,0,155721.15,0
7622,7623,15796413,Green,794,France,Male,46,6,0.00,2,1,0,195325.74,0
5392,5393,15710012,Bowen,738,Spain,Male,44,2,0.00,2,1,0,43018.82,1
7482,7483,15750104,Chan,718,Germany,Male,43,5,132615.73,2,1,0,32999.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4308,4309,15611699,Tao,641,France,Female,40,7,0.00,1,1,0,126996.67,0
8990,8991,15729065,Mackay,784,Germany,Male,28,2,109960.06,2,1,1,170829.87,0
6810,6811,15642996,Tsai,546,Germany,Female,42,9,86351.85,2,1,0,57380.13,0
6499,6500,15702561,Dale,782,France,Male,32,9,0.00,1,1,1,87566.97,0


In [43]:
df_test = pd.get_dummies(data_test)

df_test

,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,...,Surname_Walsh,Surname_Woods,Surname_Y?,Surname_Y?an,Surname_Yeh,Geography_France,Geography_Germany,Geography_Spain,Gender_Female,Gender_Male
2685,2686,15672115,679,60,6,0.00,2,1,1,77331.77,...,0,0,0,0,0,1,0,0,0,1
8017,8018,15631406,459,50,5,109387.90,1,1,0,155721.15,...,0,0,0,0,0,0,1,0,0,1
7622,7623,15796413,794,46,6,0.00,2,1,0,195325.74,...,0,0,0,0,0,1,0,0,0,1
5392,5393,15710012,738,44,2,0.00,2,1,0,43018.82,...,0,0,0,0,0,0,0,1,0,1
7482,7483,15750104,718,43,5,132615.73,2,1,0,32999.10,...,0,0,0,0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4308,4309,15611699,641,40,7,0.00,1,1,0,126996.67,...,0,0,0,0,0,1,0,0,1,0
8990,8991,15729065,784,28,2,109960.06,2,1,1,170829.87,...,0,0,0,0,0,0,1,0,0,1
6810,6811,15642996,546,42,9,86351.85,2,1,0,57380.13,...,0,0,0,0,0,0,1,0,1,0
6499,6500,15702561,782,32,9,0.00,1,1,1,87566.97,...,0,0,0,0,0,1,0,0,0,1


In [44]:
inputs.columns

Index(['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Geography_Germany',
       'Geography_Spain', 'Gender_Male'],
      dtype='object')

In [45]:
df_test = df_test[['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Geography_Germany',
       'Geography_Spain', 'Gender_Male']]

df_test

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
2685,679,60,6,0.00,2,1,1,77331.77,0,0,1
8017,459,50,5,109387.90,1,1,0,155721.15,1,0,1
7622,794,46,6,0.00,2,1,0,195325.74,0,0,1
5392,738,44,2,0.00,2,1,0,43018.82,0,1,1
7482,718,43,5,132615.73,2,1,0,32999.10,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...
4308,641,40,7,0.00,1,1,0,126996.67,0,0,0
8990,784,28,2,109960.06,2,1,1,170829.87,1,0,1
6810,546,42,9,86351.85,2,1,0,57380.13,1,0,0
6499,782,32,9,0.00,1,1,1,87566.97,0,0,1


In [46]:
scaler.fit(df_test)

scaled = scaler.transform(df_test)

df_test_scaled = pd.DataFrame(scaled, columns=df_test.columns)

df_test_scaled

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
0,0.245350,1.787560,0.373388,-1.129062,0.789542,0.531085,0.960769,-0.364816,-0.515580,-0.577350,0.960769
1,-2.029284,0.885208,-0.003772,0.635745,-0.855337,0.531085,-1.040833,0.993246,1.939563,-0.577350,0.960769
2,1.434364,0.524267,0.373388,-1.129062,0.789542,0.531085,-1.040833,1.679379,-0.515580,-0.577350,0.960769
3,0.855366,0.343796,-1.135251,-1.129062,0.789542,0.531085,-1.040833,-0.959273,-0.515580,1.732051,0.960769
4,0.648581,0.253561,-0.003772,1.010491,0.789542,0.531085,-1.040833,-1.132860,1.939563,-0.577350,0.960769
...,...,...,...,...,...,...,...,...,...,...,...
95,-0.147541,-0.017145,0.750548,-1.129062,-0.855337,0.531085,-1.040833,0.495607,-0.515580,-0.577350,-1.040833
96,1.330971,-1.099968,-1.135251,0.644976,0.789542,0.531085,0.960769,1.254998,1.939563,-0.577350,0.960769
97,-1.129769,0.163326,1.504868,0.264093,0.789542,0.531085,-1.040833,-0.710469,1.939563,-0.577350,-1.040833
98,1.310293,-0.739027,1.504868,-1.129062,-0.855337,0.531085,0.960769,-0.187495,-0.515580,-0.577350,0.960769


In [47]:
df_test['predicted_churn'] = ann.predict(df_test_scaled)

df_test

4/4 [==============================] - 0s 981us/step


/var/folders/br/yvkmbr3121n8fd9q6wdmyqfm0000gp/T/ipykernel_10910/2244387860.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test['predicted_churn'] = ann.predict(df_test_scaled)


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male,predicted_churn
2685,679,60,6,0.00,2,1,1,77331.77,0,0,1,0.036999
8017,459,50,5,109387.90,1,1,0,155721.15,1,0,1,0.709992
7622,794,46,6,0.00,2,1,0,195325.74,0,0,1,0.027448
5392,738,44,2,0.00,2,1,0,43018.82,0,1,1,0.060752
7482,718,43,5,132615.73,2,1,0,32999.10,1,0,1,0.169466
...,...,...,...,...,...,...,...,...,...,...,...,...
4308,641,40,7,0.00,1,1,0,126996.67,0,0,0,0.395852
8990,784,28,2,109960.06,2,1,1,170829.87,1,0,1,0.023840
6810,546,42,9,86351.85,2,1,0,57380.13,1,0,0,0.224618
6499,782,32,9,0.00,1,1,1,87566.97,0,0,1,0.138397
